In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_partsupp(data_folder: str, **storage_options) -> pd.DataFrame:
    # Use memory_map to speed up I/O and disable chunked dtype inference
    return pd.read_csv(
        f"{data_folder}/partsupp.tbl",
        sep="|",
        memory_map=True,
        low_memory=False
    )

partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Filter German suppliers
supplier_keys = supplier.loc[supplier["S_NATIONKEY"] == "GERMANY", "S_SUPPKEY"]

# Compute total cost per part for those suppliers and aggregate
total = (
    partsupp
      .loc[partsupp["PS_SUPPKEY"].isin(supplier_keys), ["PS_PARTKEY", "PS_SUPPLYCOST", "PS_AVAILQTY"]]
      .assign(TOTAL_COST=lambda df: df["PS_SUPPLYCOST"] * df["PS_AVAILQTY"])
      .groupby("PS_PARTKEY", as_index=False, sort=False)
      .agg(VALUE=("TOTAL_COST", "sum"))
)

# Apply threshold and sort
sum_val = total["VALUE"].sum() * 0.0001
total = total[total["VALUE"] > sum_val].sort_values("VALUE", ascending=False)